# LeetCode 1995 — Count Special Quadruplets bằng Python

**Link đề:** https://leetcode.com/problems/count-special-quadruplets/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, Enumeration

```python
class Solution:
    def countQuadruplets(self, nums: List[int]) -> int:

```

## Mục tiêu học
1. Hiểu đề: đếm bộ chỉ số `a < b < c < d` với `nums[a] + nums[b] + nums[c] == nums[d]` (đếm **theo vị trí**, không theo giá trị).
2. Viết vét cạn 4 vòng lặp lồng nhau `O(n⁴)` và hiểu vì sao nó **được phép** với `n ≤ 50`.
3. Học `collections.Counter` (từ điển đếm) và kỹ thuật **"đếm phần còn thiếu"** giống Two Sum.
4. Tối ưu dần: `O(n³)` (đếm `d` bằng Counter) → `O(n²)` (tách thành `nums[a] + nums[b] == nums[d] - nums[c]`).
5. Trace, chứng minh không đếm trùng/không sót, test đối chiếu cả 4 cách, bài tập biến thể.

## 1. Đề bài yêu cầu gì?

Cho mảng `nums` (chỉ số từ 0). Đếm số bộ **chỉ số phân biệt** `(a, b, c, d)` sao cho:
- `nums[a] + nums[b] + nums[c] == nums[d]`
- `a < b < c < d`

| `nums` | Kết quả | Các bộ |
|---|:-:|---|
| `[1,2,3,6]` | `1` | (0,1,2,3): 1+2+3 = 6 |
| `[3,3,6,4,5]` | `0` | không có |
| `[1,1,1,3,5]` | `4` | (0,1,2,3): 1+1+1=3; (0,1,3,4), (0,2,3,4), (1,2,3,4): 1+1+3=5 |

**Constraints:** `4 <= nums.length <= 50`, `1 <= nums[i] <= 100` → mọi số **dương**.

Chú ý: ví dụ 3 có 3 bộ cùng giá trị `1+1+3=5` nhưng **khác vị trí** → đếm 3 lần.

In [1]:
nums = [1, 1, 1, 3, 5]
for (a, b_, c, d) in [(0, 1, 2, 3), (0, 1, 3, 4), (0, 2, 3, 4), (1, 2, 3, 4)]:
    print(f"({a},{b_},{c},{d}): {nums[a]} + {nums[b_]} + {nums[c]} = {nums[a]+nums[b_]+nums[c]} == nums[d] = {nums[d]}")

(0,1,2,3): 1 + 1 + 1 = 3 == nums[d] = 3
(0,1,3,4): 1 + 1 + 3 = 5 == nums[d] = 5
(0,2,3,4): 1 + 1 + 3 = 5 == nums[d] = 5
(1,2,3,4): 1 + 1 + 3 = 5 == nums[d] = 5


## 2. Nền tảng Python cần biết

### 2.1. Vòng lặp lồng nhau với điều kiện `a < b < c < d`

Để đảm bảo thứ tự tăng, vòng trong bắt đầu **ngay sau** vòng ngoài: `for b in range(a + 1, n)`. Như vậy mỗi bộ chỉ số được sinh đúng **một lần**.

In [2]:
n = 5
quads = []
for a in range(n):
    for b_ in range(a + 1, n):
        for c in range(b_ + 1, n):
            for d in range(c + 1, n):
                quads.append((a, b_, c, d))
print("số bộ a<b<c<d với n=5:", len(quads))
print(quads)

số bộ a<b<c<d với n=5: 5
[(0, 1, 2, 3), (0, 1, 2, 4), (0, 1, 3, 4), (0, 2, 3, 4), (1, 2, 3, 4)]


Số bộ là $\binom{n}{4}$. Với `n = 50`: $\binom{50}{4} = 230300$ — máy tính xử lý trong tích tắc. Đó là lý do hint nói *"N is very small"*.

### 2.2. `itertools.combinations(range(n), 4)`

Cách viết gọn để sinh mọi bộ chỉ số tăng dần, tương đương 4 vòng lồng nhau ở trên.

In [3]:
from itertools import combinations
from math import comb

print(list(combinations(range(5), 4)))
print("C(50, 4) =", comb(50, 4))

[(0, 1, 2, 3), (0, 1, 2, 4), (0, 1, 3, 4), (0, 2, 3, 4), (1, 2, 3, 4)]
C(50, 4) = 230300


### 2.3. `collections.Counter` — từ điển đếm

`Counter` là `dict` đặc biệt: key là giá trị, value là **số lần xuất hiện**. Điểm tiện: truy cập key chưa có trả `0` (không lỗi `KeyError`).

In [4]:
from collections import Counter

cnt = Counter()
cnt[5] += 1
cnt[5] += 1
cnt[3] += 1
print(cnt)            # Counter({5: 2, 3: 1})
print(cnt[5], cnt[3], cnt[100])   # 2 1 0  <- key 100 chưa có vẫn trả 0

d = {}
try:
    d[100]
except KeyError:
    print("dict thường: KeyError khi key chưa có")

Counter({5: 2, 3: 1})
2 1 0
dict thường: KeyError khi key chưa có


### 2.4. `range` đi lùi — duyệt từ phải sang trái

`range(n - 1, -1, -1)` → `n-1, n-2, ..., 0`. Ta sẽ cần duyệt lùi để "gom" các phần tử bên phải vào Counter trước.

In [5]:
print(list(range(4, -1, -1)))
print(list(range(3, 0, -1)))

[4, 3, 2, 1, 0]
[3, 2, 1]


## 3. Cách 1 — Vét cạn 4 vòng lặp `O(n⁴)`

Ý tưởng trực tiếp theo đề (hint 2: *"Can we check every possible quadruplet?"*).

In [6]:
def count_quads_brute(nums):
    n = len(nums)
    count = 0
    for a in range(n):
        for b in range(a + 1, n):
            for c in range(b + 1, n):
                for d in range(c + 1, n):
                    if nums[a] + nums[b] + nums[c] == nums[d]:
                        count += 1
    return count

print(count_quads_brute([1, 2, 3, 6]))      # 1
print(count_quads_brute([3, 3, 6, 4, 5]))   # 0
print(count_quads_brute([1, 1, 1, 3, 5]))   # 4

1
0
4


### Code nộp LeetCode — Cách 1 (được Accept vì n ≤ 50)

In [7]:
from typing import List

class SolutionBrute:
    def countQuadruplets(self, nums: List[int]) -> int:
        n, count = len(nums), 0
        for a in range(n):
            for b in range(a + 1, n):
                for c in range(b + 1, n):
                    for d in range(c + 1, n):
                        if nums[a] + nums[b] + nums[c] == nums[d]:
                            count += 1
        return count

print(SolutionBrute().countQuadruplets([1, 1, 1, 3, 5]))

4


### Nút thắt

Với `n = 50` thì ổn, nhưng nếu `n = 1000`: $\binom{1000}{4} \approx 4.1 \times 10^{10}$ bộ → quá chậm.

Quan sát: vòng `d` trong cùng chỉ đang **đi tìm** các `d > c` có `nums[d] == s` (với `s = nums[a] + nums[b] + nums[c]`). Đây chính là câu hỏi **"giá trị s xuất hiện bao nhiêu lần bên phải c?"** — một câu hỏi mà **Counter** trả lời trong `O(1)`.

In [8]:
for n in [50, 200, 1000]:
    print(f"n = {n:>4}: C(n,4) = {comb(n, 4):,}")

n =   50: C(n,4) = 230,300
n =  200: C(n,4) = 64,684,950
n = 1000: C(n,4) = 41,417,124,750


## 4. Cách 2 — `O(n³)`: đếm `d` bằng Counter

Ý tưởng:
- Duyệt `c` **từ phải sang trái**. Giữ `right = Counter` các giá trị `nums[d]` với `d > c`.
- Với mỗi `c`, duyệt mọi `a < b < c`, cộng `right[nums[a] + nums[b] + nums[c]]` vào đáp án.
- Sau khi xử lý xong `c`, thêm `nums[c]` vào `right` (vì với `c` tiếp theo nhỏ hơn, vị trí `c` hiện tại trở thành một `d` hợp lệ).

Đây là phiên bản "**đếm phần còn thiếu**" giống Two Sum: biết `s` rồi, hỏi xem đã gặp bao nhiêu phần tử bằng `s`.

In [9]:
def count_quads_n3(nums):
    n = len(nums)
    right = Counter()          # giá trị nums[d] với d > c
    count = 0
    for c in range(n - 1, 1, -1):          # c = n-1 .. 2
        for a in range(c):
            for b in range(a + 1, c):
                count += right[nums[a] + nums[b] + nums[c]]
        right[nums[c]] += 1                # c trở thành ứng viên d cho các c nhỏ hơn
    return count

print(count_quads_n3([1, 2, 3, 6]))
print(count_quads_n3([3, 3, 6, 4, 5]))
print(count_quads_n3([1, 1, 1, 3, 5]))

1
0
4


### Vì sao `range(n - 1, 1, -1)`?

`c` cần ít nhất 2 chỉ số phía trước (`a`, `b`), nên `c ≥ 2`. Ở `c = n − 1` thì `right` còn rỗng (không có `d`), không đếm được gì nhưng vẫn phải chạy để **thêm `nums[n-1]`** vào `right`.

## 5. Insight cho `O(n²)` — Chuyển vế, ghép đôi hai cặp

Biến đổi phương trình:

$$nums[a] + nums[b] + nums[c] = nums[d] \iff nums[a] + nums[b] = nums[d] - nums[c]$$

Bây giờ bài toán là **ghép một cặp trái `(a, b)` với một cặp phải `(c, d)`** sao cho tổng cặp trái = hiệu cặp phải, với ràng buộc `b < c`.

Kỹ thuật: duyệt `b` **từ phải sang trái**. Trước khi xử lý `b`, ta thêm vào Counter `diff` mọi hiệu `nums[d] - nums[c]` với `c = b + 1` và `d > c`. Như vậy khi đứng ở `b`, `diff` chứa **đúng** các cặp `(c, d)` có `c > b`.

In [10]:
# Minh họa trên [1, 1, 1, 3, 5]: các hiệu nums[d] - nums[c] với từng c
nums = [1, 1, 1, 3, 5]
for c in range(len(nums)):
    diffs = [(c, d, nums[d] - nums[c]) for d in range(c + 1, len(nums))]
    print(f"c={c}: (c, d, hiệu) = {diffs}")

c=0: (c, d, hiệu) = [(0, 1, 0), (0, 2, 0), (0, 3, 2), (0, 4, 4)]
c=1: (c, d, hiệu) = [(1, 2, 0), (1, 3, 2), (1, 4, 4)]
c=2: (c, d, hiệu) = [(2, 3, 2), (2, 4, 4)]
c=3: (c, d, hiệu) = [(3, 4, 2)]
c=4: (c, d, hiệu) = []


## 6. Cách 3 — `O(n²)` (lời giải chính)

Giảng từng đoạn:

```python
diff = Counter()                    # hiệu nums[d]-nums[c] của các cặp (c, d) với c > b
count = 0
for b in range(n - 3, 0, -1):       # b cần a phía trước (b>=1) và c, d phía sau (b<=n-3)
    c = b + 1                       # (1) cặp mới có c = b+1 vừa trở nên hợp lệ
    for d in range(c + 1, n):
        diff[nums[d] - nums[c]] += 1
    for a in range(b):              # (2) mọi a < b
        count += diff[nums[a] + nums[b]]
return count
```

Mỗi `b` thêm `O(n)` cặp và tra `O(n)` lần → tổng `O(n²)`.

In [11]:
def count_quads_n2(nums):
    n = len(nums)
    diff = Counter()
    count = 0
    for b in range(n - 3, 0, -1):
        c = b + 1
        for d in range(c + 1, n):
            diff[nums[d] - nums[c]] += 1
        for a in range(b):
            count += diff[nums[a] + nums[b]]
    return count

print(count_quads_n2([1, 2, 3, 6]))      # 1
print(count_quads_n2([3, 3, 6, 4, 5]))   # 0
print(count_quads_n2([1, 1, 1, 3, 5]))   # 4

1
0
4


## 7. TRACE từng bước (Cách 3)

Ví dụ `nums = [1, 1, 1, 3, 5]`, `n = 5` → `b` chạy `2, 1`.

| b | c = b+1 | Cặp (c,d) thêm vào | diff sau khi thêm | Các a | nums[a]+nums[b] | diff[...] | count |
|:-:|:-:|---|---|---|---|---|:-:|
| 2 | 3 | (3,4): 5−3 = 2 | {2:1} | 0, 1 | 2, 2 | 1, 1 | 2 |
| 1 | 2 | (2,3): 3−1 = 2; (2,4): 5−1 = 4 | {2:2, 4:1} | 0 | 2 | 2 | **4** |

Đối chiếu với đề: ở `b = 2` đếm được (0,2,3,4) và (1,2,3,4); ở `b = 1` đếm (0,1,2,3) [hiệu 2 của cặp (2,3)] và (0,1,3,4) [hiệu 2 của cặp (3,4)].

In [12]:
def trace_count_quads(nums):
    n = len(nums)
    diff = Counter()
    count = 0
    print(f"nums = {nums}")
    for b in range(n - 3, 0, -1):
        c = b + 1
        added = []
        for d in range(c + 1, n):
            diff[nums[d] - nums[c]] += 1
            added.append(f"({c},{d}):{nums[d]}-{nums[c]}={nums[d]-nums[c]}")
        print(f"b={b}: thêm {', '.join(added)} -> diff = {dict(diff)}")
        for a in range(b):
            need = nums[a] + nums[b]
            got = diff[need]
            count += got
            print(f"    a={a}: nums[a]+nums[b] = {nums[a]}+{nums[b]} = {need} -> diff[{need}] = {got}, count = {count}")
    print("Kết quả:", count)
    return count

trace_count_quads([1, 1, 1, 3, 5])

nums = [1, 1, 1, 3, 5]
b=2: thêm (3,4):5-3=2 -> diff = {2: 1}
    a=0: nums[a]+nums[b] = 1+1 = 2 -> diff[2] = 1, count = 1
    a=1: nums[a]+nums[b] = 1+1 = 2 -> diff[2] = 1, count = 2
b=1: thêm (2,3):3-1=2, (2,4):5-1=4 -> diff = {2: 2, 4: 1}
    a=0: nums[a]+nums[b] = 1+1 = 2 -> diff[2] = 2, count = 4
Kết quả: 4


4

In [13]:
trace_count_quads([1, 2, 3, 6])

nums = [1, 2, 3, 6]
b=1: thêm (2,3):6-3=3 -> diff = {3: 1}
    a=0: nums[a]+nums[b] = 1+2 = 3 -> diff[3] = 1, count = 1
Kết quả: 1


1

## 8. Code nộp LeetCode

In [14]:
class Solution:
    def countQuadruplets(self, nums: List[int]) -> int:
        n = len(nums)
        diff = Counter()
        count = 0
        for b in range(n - 3, 0, -1):
            c = b + 1
            for d in range(c + 1, n):
                diff[nums[d] - nums[c]] += 1
            for a in range(b):
                count += diff[nums[a] + nums[b]]
        return count

for nums, expected in [([1, 2, 3, 6], 1), ([3, 3, 6, 4, 5], 0), ([1, 1, 1, 3, 5], 4)]:
    got = Solution().countQuadruplets(nums)
    print(nums, "->", got)
    assert got == expected

[1, 2, 3, 6] -> 1
[3, 3, 6, 4, 5] -> 0
[1, 1, 1, 3, 5] -> 4


## 9. Cách 4 — Đếm theo **giá trị** nhờ `nums[i] ≤ 100` (Counter dạng mảng)

Vì giá trị nhỏ (`1..100`), thay `Counter` bằng **list** cỡ cố định — truy cập theo chỉ số nhanh hơn băm. Áp dụng cho Cách 2: `right[v]` = số `d > c` có `nums[d] = v`. Tổng `nums[a]+nums[b]+nums[c]` có thể tới 300 > 100 → phải kiểm tra trước khi tra.

Cách này minh họa ý tưởng **"khi miền giá trị nhỏ, dùng mảng đếm thay cho hash map"**.

In [15]:
def count_quads_array(nums):
    n = len(nums)
    right = [0] * 101               # right[v]: số d > c có nums[d] == v
    count = 0
    for c in range(n - 1, 1, -1):
        for a in range(c):
            for b in range(a + 1, c):
                s = nums[a] + nums[b] + nums[c]
                if s <= 100:
                    count += right[s]
        right[nums[c]] += 1
    return count

print(count_quads_array([1, 1, 1, 3, 5]))
print(count_quads_array([28, 8, 49, 85, 37, 90, 20, 8]))

4
1


## 10. Vì sao đúng? (không đếm trùng, không bỏ sót)

**Cách 3 — Invariant:** ngay trước vòng `for a` ứng với `b`, `diff[v]` = số cặp `(c, d)` với `b < c < d < n` và `nums[d] − nums[c] = v`.

- **Khởi tạo** (`b = n − 3`): ta vừa thêm các cặp có `c = n − 2`, đó là **mọi** cặp `(c, d)` với `c > n − 3`. ✓
- **Duy trì:** khi `b` giảm 1, các cặp mới hợp lệ là những cặp có `c = b + 1` (mới) — đúng là cái ta thêm. Các cặp cũ có `c > b + 1` vẫn hợp lệ. ✓

Với mỗi `a < b`, `diff[nums[a] + nums[b]]` đếm **đúng** số cặp `(c, d)` ghép được → mỗi bộ `(a, b, c, d)` thỏa được đếm **đúng một lần** (tại đúng `b` của nó, đúng `a` của nó).

**Cách 2** tương tự, invariant: trước khi xử lý `c`, `right` chứa đúng các `nums[d]` với `d > c`.

## 11. Time / Space và bảng so sánh

Gọi `n = len(nums)`, `V = 100` (giá trị lớn nhất).

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. Vét cạn | 4 vòng lặp | `O(n⁴)` | `O(1)` | Được Accept vì n ≤ 50 |
| 2. Counter cho d | duyệt `c` lùi, tra `right[s]` | `O(n³)` | `O(n)` | "đếm phần còn thiếu" |
| 3. Ghép hai cặp | `nums[a]+nums[b] == nums[d]-nums[c]` | `O(n²)` | `O(n²)` | **Tối ưu** |
| 4. Mảng đếm | như cách 2 nhưng list cỡ 101 | `O(n³)` | `O(V)` | Khi miền giá trị nhỏ |

Cách 3 dùng `O(n²)` bộ nhớ vì Counter có thể chứa tới ~n²/2 cặp (thực tế số key khác nhau ≤ số hiệu khác nhau ≤ 2V).

In [16]:
import time, random

rng = random.Random(0)
big = [rng.randint(1, 100) for _ in range(50)]
for f in (count_quads_brute, count_quads_n3, count_quads_n2):
    t = time.perf_counter()
    r = f(big)
    print(f"{f.__name__:<18} -> {r:>5} | {1000 * (time.perf_counter() - t):7.2f} ms")

count_quads_brute  ->   275 |   17.00 ms
count_quads_n3     ->   275 |    3.98 ms
count_quads_n2     ->   275 |    0.60 ms


(Thời gian đo phụ thuộc máy — chỉ để so sánh tương đối, đừng coi là con số chính xác.)

## 12. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

1. Ba ví dụ đề.
2. Biên `n = 4` đúng và sai.
3. Toàn số bằng nhau: `[1, 1, 1, 1]` → 0 (1+1+1 ≠ 1); `[1,1,1,3,3]` có trùng.
4. Phản ví dụ "thứ tự": `[6, 1, 2, 3]` → 0 (tổng đúng nhưng `d` đứng **trước**).
5. Đối chiếu ngẫu nhiên giữa 4 cách.

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<28} PASS ({actual})")

check("đề 1", count_quads_n2([1, 2, 3, 6]), 1)
check("đề 2", count_quads_n2([3, 3, 6, 4, 5]), 0)
check("đề 3", count_quads_n2([1, 1, 1, 3, 5]), 4)
check("n=4 không thỏa", count_quads_n2([1, 1, 1, 1]), 0)
check("d đứng trước -> 0", count_quads_n2([6, 1, 2, 3]), 0)
check("nhiều d trùng", count_quads_n2([1, 1, 1, 3, 3]), 2)
check("tổng > 100 vô nghĩa", count_quads_n2([100, 100, 100, 100]), 0)

đề 1                         PASS (1)
đề 2                         PASS (0)
đề 3                         PASS (4)
n=4 không thỏa               PASS (0)
d đứng trước -> 0            PASS (0)
nhiều d trùng                PASS (2)
tổng > 100 vô nghĩa          PASS (0)


In [18]:
rng = random.Random(1995)
for _ in range(800):
    nums = [rng.randint(1, 8) for _ in range(rng.randint(4, 12))]
    expected = count_quads_brute(nums)
    assert count_quads_n3(nums) == expected
    assert count_quads_n2(nums) == expected
    assert count_quads_array(nums) == expected
    assert Solution().countQuadruplets(nums) == expected
print("800 test ngẫu nhiên: 4 cách khớp với vét cạn")

800 test ngẫu nhiên: 4 cách khớp với vét cạn


Chọn giá trị nhỏ `1..8` để phương trình **hay thỏa** — nếu chọn `1..100` thì đa số test cho 0, kiểm tra kém hiệu quả.

## 13. Lỗi người mới hay gặp

### Lỗi 1: Đếm theo **giá trị** thay vì theo **chỉ số** (dùng `set`)

In [19]:
def wrong_set_values(nums):
    found = set()
    n = len(nums)
    for a, b, c, d in combinations(range(n), 4):
        if nums[a] + nums[b] + nums[c] == nums[d]:
            found.add((nums[a], nums[b], nums[c], nums[d]))
    return len(found)

print("sai :", wrong_set_values([1, 1, 1, 3, 5]))   # 2
print("đúng:", count_quads_n2([1, 1, 1, 3, 5]))     # 4

sai : 2
đúng: 4


**Vì sao sai:** đề đếm bộ **chỉ số** phân biệt; `(0,1,3,4)` và `(0,2,3,4)` cùng giá trị nhưng là hai bộ khác nhau. **Sửa:** đếm mỗi bộ chỉ số, không loại trùng giá trị.

### Lỗi 2: Cách 3 — lấy `c = b` thay vì `c = b + 1`

In [20]:
def wrong_c_is_b(nums):
    n = len(nums)
    diff = Counter()
    count = 0
    for b in range(n - 2, 0, -1):
        c = b                               # sai: c phải > b
        for d in range(c + 1, n):
            diff[nums[d] - nums[c]] += 1
        for a in range(b):
            count += diff[nums[a] + nums[b]]
    return count

print("sai :", wrong_c_is_b([1, 1, 1, 3, 5]))
print("đúng:", count_quads_n2([1, 1, 1, 3, 5]))

sai : 7
đúng: 4


**Vì sao sai:** khi `c = b`, cặp `(c, d)` dùng **chung chỉ số** với cặp `(a, b)` — một phần tử bị dùng hai lần, đếm thừa. **Sửa:** `c = b + 1`, và `b` chạy tới `n − 3`.

*Ghi chú:* ở Cách 2, nếu lỡ thêm `nums[c]` vào `right` **trước** khi tra, bài này tình cờ vẫn đúng vì mọi số đều dương nên `nums[a] + nums[b] + nums[c] > nums[c]` — không bao giờ khớp với chính `c`. Đừng dựa vào may mắn đó: nếu đề cho phép số 0 hoặc số âm thì sẽ sai. Luôn **tra trước, thêm sau**.

### Lỗi 3: Cách 3 — thêm **mọi** cặp `(c, d)` ngay từ đầu (không ràng buộc `b < c`)

In [21]:
def wrong_all_pairs(nums):
    n = len(nums)
    diff = Counter(nums[d] - nums[c] for c in range(n) for d in range(c + 1, n))
    return sum(diff[nums[a] + nums[b]] for a in range(n) for b in range(a + 1, n))

print("sai :", wrong_all_pairs([1, 1, 1, 3, 5]))
print("đúng:", count_quads_n2([1, 1, 1, 3, 5]))

sai : 21
đúng: 4


**Vì sao sai:** ghép cả những cặp `(c, d)` nằm **trước** hoặc **chồng lên** `(a, b)` — vi phạm `b < c`. **Sửa:** duyệt `b` lùi và chỉ thêm cặp có `c = b + 1` mỗi bước.

### Lỗi 4: Dùng `dict` thường và truy cập key chưa có
`diff[x]` với `dict` thường ném `KeyError`. **Sửa:** dùng `Counter`, hoặc `diff.get(x, 0)`.

## 14. Mẹo nhận dạng pattern

- Ràng buộc nhỏ (`n ≤ 50`) → vét cạn là lựa chọn hợp lệ; đừng ngại.
- "Đếm số bộ thỏa phương trình tổng" → **chuyển vế** để tách thành hai nửa độc lập, dùng **hash map đếm** (meet in the middle). Ví dụ: LeetCode 454 — 4Sum II.
- Ràng buộc thứ tự chỉ số `a < b < c < d` + hash map → **duyệt điểm chia** (ở đây là `b`) và cập nhật map dần dần để map chỉ chứa phần hợp lệ.
- Miền giá trị nhỏ → mảng đếm thay hash map.

## 15. Biến thể

1. **Đếm bộ ba** `i < j < k` với `nums[i] + nums[j] == nums[k]`.
2. **LeetCode 454 — 4Sum II:** bốn mảng, đếm `A[i]+B[j]+C[k]+D[l] == 0` (không có ràng buộc thứ tự → tách hai nửa trực tiếp).
3. Đếm bộ bốn với `nums[a] * nums[b] == nums[c] * nums[d]` (LeetCode 1726 — Tuple with Same Product).

In [22]:
def count_triplets_sum(nums):
    # nums[i] + nums[j] == nums[k], i < j < k: duyệt j lùi, right đếm nums[k] với k > j
    right = Counter()
    count = 0
    for j in range(len(nums) - 1, 0, -1):
        for i in range(j):
            count += right[nums[i] + nums[j]]
        right[nums[j]] += 1
    return count

def four_sum_count(A, B, C, D):
    ab = Counter(x + y for x in A for y in B)
    return sum(ab[-(z + w)] for z in C for w in D)

print("bộ ba [1,1,2,3]:", count_triplets_sum([1, 1, 2, 3]))
print("4Sum II:", four_sum_count([1, 2], [-2, -1], [-1, 2], [0, 2]))

bộ ba [1,1,2,3]: 3
4Sum II: 2


Kiểm tra tay bộ ba `[1,1,2,3]`: (0,1,2): 1+1=2 ✓; (0,2,3): 1+2=3 ✓; (1,2,3): 1+2=3 ✓ → **3**.

## 16. Bài tập tự luyện (TODO — chưa chạy)

### Bài 1 — Tính tay
Liệt kê mọi bộ `(a,b,c,d)` thỏa với `nums = [2, 1, 1, 4, 4]`. Đáp án là bao nhiêu?

### Bài 2 — Dự đoán output
Với `nums = [1, 1, 1, 3, 5]`, `wrong_all_pairs` cho 21 thay vì 4. Hãy chỉ ra một bộ "giả" mà nó đếm (ví dụ cặp `(c, d)` đứng trước `(a, b)`).

### Bài 3 — Tự viết vét cạn bằng `combinations`

In [23]:
def exercise_brute(nums: list[int]) -> int:
    # TODO: for a, b, c, d in combinations(range(len(nums)), 4): ...
    pass

# assert exercise_brute([1, 1, 1, 3, 5]) == 4

### Bài 4 — Tự viết lời giải `O(n²)`

In [24]:
def exercise_n2(nums: list[int]) -> int:
    # TODO 1: diff = Counter(), count = 0
    # TODO 2: for b in range(n - 3, 0, -1): thêm các hiệu nums[d] - nums[b + 1]
    # TODO 3: for a in range(b): count += diff[nums[a] + nums[b]]
    pass

# assert exercise_n2([1, 2, 3, 6]) == 1
# assert exercise_n2([1, 1, 1, 3, 5]) == 4

### Bài 5 — Biến thể: đếm bộ ba `i < j < k` với `nums[i] + nums[j] + nums[k] == target`

In [25]:
def exercise_count_three_sum(nums: list[int], target: int) -> int:
    # TODO: duyệt k, dùng Counter các nums[i] (i < j) ... hoặc cách bạn chọn, miễn O(n²)
    pass

# assert exercise_count_three_sum([1, 1, 2, 2, 3], 5) == 3
# assert exercise_count_three_sum([1, 1, 1], 3) == 1

### Bài 6 — Mini challenge: LeetCode 1726 — đếm số bộ `(a, b, c, d)` (các **giá trị** phân biệt từ mảng giá trị phân biệt) với `a*b == c*d`; mỗi lựa chọn 2 cặp cùng tích cho 8 hoán vị

In [26]:
def exercise_tuple_same_product(nums: list[int]) -> int:
    # TODO: Counter các tích nums[i]*nums[j] (i<j); với mỗi tích xuất hiện m lần: + 8 * C(m, 2)
    pass

# assert exercise_tuple_same_product([2, 3, 4, 6]) == 8
# assert exercise_tuple_same_product([1, 2, 4, 5, 10]) == 16

## 17. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | `d` phải là chỉ số 3 hoặc 4 (giá trị 4) | cần `a+b+c` = 4 từ {2,1,1,...} | 2+1+1 = 4 |
| 2 | In ra các bộ chỉ số nó dùng | thử `(a,b) = (2,3)` | `(c,d) = (0,4)`: 5−1 = 4 = 1+3 |
| 3 | `combinations(range(n), 4)` | `if ...: count += 1` | trả `count` |
| 4 | Khuôn mục 6 | `c = b + 1` | thêm trước, tra sau (trong cùng b) |
| 5 | Chuyển vế `nums[i] + nums[j] = target − nums[k]` | duyệt `j`, Counter các `nums[i]` với `i < j`… | hoặc: duyệt `j` lùi, Counter `nums[k]` với `k > j`, tra `target − nums[i] − nums[j]` |
| 6 | Đếm tích từng cặp | `m` cặp cùng tích → `C(m,2)` cách chọn 2 cặp | mỗi cách ×8 hoán vị |

## 18. Đáp án tham khảo, template và checklist

- Bài 1: `[2,1,1,4,4]`: tổng 2+1+1 = 4 với `(0,1,2)`, `d ∈ {3, 4}` → **2** bộ: (0,1,2,3), (0,1,2,4).
- Bài 2: một bộ giả: `(a,b) = (2,3)` (1 + 3 = 4) ghép với `(c,d) = (0,4)` (5 − 1 = 4) — `c = 0` đứng trước `a` → vi phạm `b < c`.

In [27]:
def answer_brute(nums):
    return sum(1 for a, b, c, d in combinations(range(len(nums)), 4)
               if nums[a] + nums[b] + nums[c] == nums[d])

def answer_n2(nums):
    n, diff, count = len(nums), Counter(), 0
    for b in range(n - 3, 0, -1):
        c = b + 1
        for d in range(c + 1, n):
            diff[nums[d] - nums[c]] += 1
        for a in range(b):
            count += diff[nums[a] + nums[b]]
    return count

def answer_count_three_sum(nums, target):
    right, count = Counter(), 0
    for j in range(len(nums) - 1, 0, -1):
        for i in range(j):
            count += right[target - nums[i] - nums[j]]
        right[nums[j]] += 1
    return count

def answer_tuple_same_product(nums):
    prod = Counter(nums[i] * nums[j] for i in range(len(nums)) for j in range(i + 1, len(nums)))
    return sum(8 * comb(m, 2) for m in prod.values())

assert answer_brute([2, 1, 1, 4, 4]) == 2
assert answer_brute([1, 1, 1, 3, 5]) == 4
assert answer_n2([1, 2, 3, 6]) == 1 and answer_n2([1, 1, 1, 3, 5]) == 4
assert answer_count_three_sum([1, 1, 2, 2, 3], 5) == 3
assert answer_count_three_sum([1, 1, 1], 3) == 1
assert answer_tuple_same_product([2, 3, 4, 6]) == 8
assert answer_tuple_same_product([1, 2, 4, 5, 10]) == 16
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


### Template nhớ nhanh

```python
class Solution:
    def countQuadruplets(self, nums: List[int]) -> int:
        n, diff, count = len(nums), Counter(), 0
        for b in range(n - 3, 0, -1):
            c = b + 1
            for d in range(c + 1, n):          # cặp (c, d) mới hợp lệ
                diff[nums[d] - nums[c]] += 1
            for a in range(b):                 # tra tổng cặp trái
                count += diff[nums[a] + nums[b]]
        return count
```

Câu thần chú: **"Chuyển vế → hai cặp → duyệt điểm chia lùi → Counter."**

### Checklist trước khi submit
- [ ] Đếm theo **chỉ số**, không loại trùng giá trị?
- [ ] Counter chỉ chứa phần tử/cặp **hợp lệ về thứ tự** tại thời điểm tra?
- [ ] Thứ tự "thêm" và "tra" đúng (Cách 2: tra rồi thêm; Cách 3: thêm cặp `c = b+1` rồi tra)?
- [ ] Biên `range(n - 3, 0, -1)` cho `b` (cần `a` trước, `c, d` sau)?
- [ ] Đã test ví dụ có giá trị trùng `[1,1,1,3,5]` → 4?